In [8]:
# For Local Work. Don't export next cell if you want to run it locally
import requests
import pandas as pd
import json
from dotenv import load_dotenv
import os
load_dotenv()
import sys

In [9]:
# Load this if you are in Google Colab, i export stuff with it
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# NASA Data Export

In [11]:
# Credentials and path to file depending on an environment
if 'google' in sys.modules:
    credentials = userdata.get('NASA_ACCESS_KEY')
    csv_file_path = '/content/drive/MyDrive/temp_colab_data/all_neos.csv'
else:
    credentials = os.getenv('NASA_ACCESS_KEY')
    csv_file_path = '../data/nasa_data/all_neos.csv'

# Setting up a batch size. I know this is not the best way to do it, but it works without any issues

url = 'https://api.nasa.gov/neo/rest/v1/neo/browse?api_key=' + credentials
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    print(f"Initial request failed with status code: {response.status_code}")
    sys.exit(0)

# Now we are exporting all NEOs to a csv file
for _ in range(data['page']['total_pages']):
    # We have to be able to stop our script at any point if it crashes or something and continue from where we left off
    try:
        if _ in pd.read_csv(csv_file_path, usecols=['page'])['page'].values:
            print('skipped',_)
            continue
    except:
        pass

    # Creating a link
    url = f'https://api.nasa.gov/neo/rest/v1/neo/browse?page={_}&size=20&api_key=' + credentials
    print('loading page ' + str(_))

    # It's just more understandable this way cmon
    response = requests.get(url)
    data = response.json()

    filtered_data = pd.DataFrame(data['near_earth_objects']).drop('links', axis=1) # Dropping the links column as a security concern
    filtered_data['page'] = _ #Adding a page column to the dataframe
    filtered_data = filtered_data[['id', 'neo_reference_id', 'name', 'designation', 'nasa_jpl_url',
       'absolute_magnitude_h', 'estimated_diameter',
       'is_potentially_hazardous_asteroid', 'close_approach_data',
       'orbital_data', 'is_sentry_object', 'page']] # Using fixed list of columns in a fixed order

    # Exporting the data
    if response.status_code == 200:
        if _ == 0: # If it's the first page, we write the header and creating a file basically
            filtered_data.to_csv(csv_file_path, mode='w', index=False, header=True)
        else:
            filtered_data.to_csv(csv_file_path, mode='a', index=False, header=False)
    else:
        print(f"Request failed with status code: {response.status_code}")
        break

loading page 0


IsADirectoryError: [Errno 21] Is a directory: '/content/drive/MyDrive/temp_colab_data'